In [4]:
%pwd

'/home/jongha/projects/mlops'

In [5]:
"""
< 허깅페이스 3형제 >

1. datasets
2. transformers
3. huggingface_hub

이 셋은 같은 가문이지만 서로 속해있지 않으므로 주의
"""


###########################################
# # 1. 데이터를 다루고 싶을 때 (Label 정보 등)
###########################################
"""
import datasets 
from datasets import load_dataset
"""


####################################################
# # 2. 모델 아키텍처나 토크나이저를 다루고 싶을 때
####################################################
"""
import transformers
from transformers import AutoModel, AutoTokenizer
"""


##################################################
# # 3. 허깅페이스 서버 설정이나 로그인을 다루고 싶을 때
##################################################
"""
import huggingface_hub
from huggingface_hub import login
"""


###################################################
# # 허깅페이스 토큰 로긴 방법
# # huggingface_hub 라이브러리 설치 필요
###################################################
"""uv pip install huggingface_hub"""


##############################
# # 로그인 명령 실행 (터미널에서)
##############################
"""huggingface-cli login"""

'huggingface-cli login'

#### 모델 빌드

In [6]:
from datasets import load_dataset


ag_news_dataset = load_dataset(path="ag_news")

print(f"ag_news_dataset: \n{ag_news_dataset}\n")


news_labels = ag_news_dataset["train"].features["label"].names

print(f"news_labels: \n{news_labels}\n")


id2label = {i: label for i, label in enumerate(news_labels)}
label2id = {label: i for i, label in enumerate(news_labels)}

print(f"id2label: \n{id2label}\n")
print(f"label2id: \n{label2id}\n")

ag_news_dataset: 
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

news_labels: 
['World', 'Sports', 'Business', 'Sci/Tech']

id2label: 
{0: 'World', 1: 'Sports', 2: 'Business', 3: 'Sci/Tech'}

label2id: 
{'World': 0, 'Sports': 1, 'Business': 2, 'Sci/Tech': 3}



In [7]:
from transformers import BertTokenizer, BatchEncoding, BertForSequenceClassification


tokenizer: BertTokenizer = BertTokenizer.from_pretrained(
    pretrained_model_name_or_path="bert-base-uncased",
)

model: BertForSequenceClassification = BertForSequenceClassification.from_pretrained(
    pretrained_model_name_or_path="bert-base-uncased",
    num_labels=4,
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3066.57it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider tr

In [8]:
train_dataset = ag_news_dataset["train"]

print(f"{train_dataset}\n")

test_dataset = ag_news_dataset["test"]

print(test_dataset)

Dataset({
    features: ['text', 'label'],
    num_rows: 120000
})

Dataset({
    features: ['text', 'label'],
    num_rows: 7600
})


In [9]:
train_dataset_text = list(train_dataset["text"])

print(f"train[0]: \n{train_dataset_text[0]}\n")

test_dataset_text = list(test_dataset["text"])

print(f"test[0]: \n{test_dataset_text[0]}")

train[0]: 
Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.

test[0]: 
Fears for T N pension after talks Unions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul.


#### <번외> - 데이터셋 메서드의 활용

In [10]:
def preprocess_data(dataset) -> BatchEncoding:
    preprocessed_data = tokenizer(
        text=dataset["text"],
        truncation=True,
        padding="max_length",
        max_length=512,
        # return_tensors="pt"
    )

    return preprocessed_data


preprocessed_train = train_dataset.select(range(2000)).map(preprocess_data, batched=True)
preprocessed_test = test_dataset.select(range(200)).map(preprocess_data, batched=True)

print(preprocessed_train)
print(preprocessed_test)

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 2000
})
Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 200
})


In [11]:
formatted_train = preprocessed_train.with_format(type="torch", columns=["input_ids", "token_type_ids", "attention_mask", "label"])
formatted_test = preprocessed_test.with_format(type="torch", columns=["input_ids", "token_type_ids", "attention_mask", "label"])

print(formatted_train)
print(formatted_test)

relocated_train = formatted_train
relocated_test = formatted_test

print(len(relocated_train))
print(len(relocated_test))

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 2000
})
Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 200
})
2000
200


#### 데이터 재배치

In [12]:
truncated_train = train_dataset_text[:2000]
truncated_test = test_dataset_text[:200]
truncated_train_label = ag_news_dataset["train"][:2000]["label"]
truncated_test_label = ag_news_dataset["test"][:200]["label"]

print(f"truncated_train_label[0]: {truncated_train_label[0]}\n")
print(f"truncated_test_label[0]: {truncated_test_label[0]}\n")


preprocessed_train_dataset = tokenizer(
    truncated_train,
    padding=True,
    max_length=512,
    truncation=True,
    return_tensors="pt",
)
preprocessed_test_dataset = tokenizer(
    truncated_test,
    padding=True,
    max_length=512,
    truncation=True,
    return_tensors="pt",
)

print(preprocessed_train_dataset)
print(preprocessed_test_dataset)

truncated_train_label[0]: 2

truncated_test_label[0]: 2

{'input_ids': tensor([[  101,  2813,  2358,  ...,     0,     0,     0],
        [  101, 18431,  2571,  ...,     0,     0,     0],
        [  101,  3514,  1998,  ...,     0,     0,     0],
        ...,
        [  101,  4238, 17016,  ...,     0,     0,     0],
        [  101, 11844,  6715,  ...,     0,     0,     0],
        [  101,  6645, 11372,  ...,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])}
{'input_ids': tensor([[  101, 10069,  2005,  ...,     0,     0,     0],
        [  101,  199

In [13]:
import torch


relocated_train = []
for i in range(len(truncated_train)):
    relocated_train.append({key: tensor[i] for key, tensor in preprocessed_train_dataset.items()})

for i in range(len(relocated_train)):
    relocated_train[i].update({"labels": torch.tensor(truncated_train_label[i])})

print(f"길이: {len(relocated_train)}")
print(relocated_train[0].keys())


relocated_test = []
for i in range(len(truncated_test)):
    relocated_test.append({key: tensor[i] for key, tensor in preprocessed_test_dataset.items()})

for i in range(len(truncated_test)):
    relocated_test[i].update({"labels": torch.tensor(truncated_test_label[i])})
    
print(f"길이: {len(relocated_test)}")
print(relocated_test[0].keys())

길이: 2000
dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'labels'])
길이: 200
dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'labels'])


#### 데이터 로더

In [14]:
from torch.utils.data import DataLoader


train_loader = DataLoader(
    dataset=relocated_train,
    batch_size=16,
    shuffle=True,
    pin_memory=True,
)
test_loader = DataLoader(
    dataset=relocated_test,
    batch_size=16,
)

print(train_loader)
print(test_loader)

#### 하이퍼 파라미터

In [16]:
import torch
from torch.optim import AdamW
from torch import nn


num_epochs = 5
optimizer = AdamW(
    params=model.parameters(),
    lr=5e-5,
    weight_decay=0.01,
)
criterion: nn.CrossEntropyLoss = nn.CrossEntropyLoss()
device = torch.device(device="cuda" if torch.cuda.is_available() else "cpu")

print(f"cuda: {torch.cuda.is_available()}")

cuda: True


In [17]:
from uu import Error


raise Error

/tmp/ipykernel_44760/3102472954.py:1: DeprecationWarning: 'uu' is deprecated and slated for removal in Python 3.13
  from uu import Error


Error: 

#### 모델 학습

In [ ]:
from tqdm import tqdm


model.to(device)

for epoch in range(num_epochs):
    model.train()    
    losses = []
    for batch in tqdm(train_loader):
        optimizer.zero_grad()
        inputs = {
            "input_ids": batch["input_ids"].to(device),
            "token_type_ids": batch["token_type_ids"].to(device),
            "attention_mask": batch["attention_mask"].to(device),
            "labels": batch["label"].to(device),
        }
        outputs = model(**inputs)
        loss = outputs.loss
        losses.append(loss.item())
        loss.backward()
        optimizer.step()

        average_loss = (sum(losses) / len(train_loader))        

    print(f"에포크 {epoch + 1} 완료, 평균 손실: {average_loss:.2f}")

outputs

100%|██████████| 125/125 [00:41<00:00,  2.98it/s]


에포크 1 완료, 평균 손실: 0.62


100%|██████████| 125/125 [00:41<00:00,  2.98it/s]


에포크 2 완료, 평균 손실: 0.29


100%|██████████| 125/125 [00:41<00:00,  3.01it/s]


...

100%|██████████| 125/125 [00:41<00:00,  2.98it/s]


에포크 4 완료, 평균 손실: 0.10


100%|██████████| 125/125 [00:42<00:00,  2.97it/s]


에포크 5 완료, 평균 손실: 0.05


SequenceClassifierOutput(loss=tensor(0.0094, device='cuda:0', grad_fn=<NllLossBackward0>), logits=tensor([[-2.4570, -2.0342, -0.5218,  5.4174],
        [-1.5102,  4.9392, -1.3219, -2.1935],
        [-1.5398,  5.4205, -1.9244, -2.0518],
        [-0.9889, -1.6782,  5.0139, -1.8677],
        [-1.7801,  5.6020, -1.6570, -2.3324],
        [ 3.9988, -0.2310, -2.2209, -1.8284],
        [-1.5884,  5.4879, -1.8599, -2.4343],
        [-2.0006, -1.5718, -1.2015,  4.7611],
        [-1.4899,  5.1586, -1.9655, -2.2159],
        [-0.3150, -1.7294,  4.9220, -2.6970],
        [-2.1250, -2.5922, -0.4031,  5.2589],
        [-1.6497,  5.6204, -1.7905, -2.4968],
        [-2.7074, -1.7067, -0.3193,  4.7864],
        [-1.0610,  5.4646, -1.9854, -2.4410],
        [-2.9598, -2.0834,  1.5369,  4.1030],
        [ 5.0771, -2.3032, -1.1365, -2.0115]], device='cuda:0',
       grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)

#### 모델 저장

In [ ]:
state_dict = model.state_dict()
torch.save(
    obj=state_dict,
    f="./models/bert/bert_news_classifier_v2.pth"
)

#### 모델 로드 및 테스트

In [19]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification


tokenizer: BertTokenizer = BertTokenizer.from_pretrained(
    pretrained_model_name_or_path="bert-base-uncased",
)
model: BertForSequenceClassification = BertForSequenceClassification.from_pretrained(
    pretrained_model_name_or_path="bert-base-uncased",
    num_labels=4,
)


state_dict = torch.load(
    f="./models/bert/bert_news_classifier_v2.pth",
    weights_only=True,
)
model.load_state_dict(
    state_dict=state_dict,
)
model.eval()
model.to(device)


corrects = []
for batch in test_loader:
    batch = {key: batch[key].to(device) for key, tensor in batch.items()}
    # labels = batch.pop("label")
    labels = batch.pop("labels")
    with torch.no_grad():
        outputs = model(**batch)
        logits = outputs.logits
        model_predictions = torch.softmax(input=logits, dim=1).argmax(dim=1)
        corrects.append(sum(model_predictions == labels))        

accuracy = (sum(corrects) / len(relocated_test))

print(accuracy)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2971.16it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider tr

tensor(0.9300, device='cuda:0')


#### 샘플 투입

In [20]:
import json


with open(file="data/request_sample_json/batch_request.json", mode="r", encoding="utf-8") as file:
    text = json.load(file)

text

{'data': ['DETROIT – America maintained its love affair with pickup trucks in 2023 — but a top-selling vehicle from Toyota Motor nearly ruined their tailgate party.Sales of the Toyota RAV4 compact crossover came within 10,000 units of Stellantis’ Ram pickup truck last year, a near-No. 3 ranking that would have marked the first time since 2014 that a non-pickup claimed one of the top three U.S. sales podium positions.The RAV4 has rapidly closed the gap: In 2020, the vehicle undersold the Ram truck by more than 133,000 units. Last year, it lagged by just 9,983. Stellantis sold 444,926 Ram pickups last year, a 5% decline from 2022.“Trucks are always at the top because they’re bought by not only individuals, but also fleet buyers and we saw heavy fleet buying last year,” said Michelle Krebs, an executive analyst at Cox Automotive. “The RAV4 shows that people want affordable, smaller SUVs, and the fact that there’s also a hybrid version of that makes it popular with people.”',
  'OpenVoice 

In [21]:
preprocessed_data = tokenizer(
    text=text["data"],
    truncation=True,
    padding=True,
    max_length=512,
    return_tensors="pt",
)

print(type(preprocessed_data))
print(preprocessed_data)


<class 'transformers.tokenization_utils_base.BatchEncoding'>
{'input_ids': tensor([[  101,  5626,  1516,  ...,     0,     0,     0],
        [  101,  2330,  6767,  ...,     0,     0,     0],
        [  101,  1038, 19738,  ...,  1037,  2192,   102]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1]])}


In [36]:
preprocessed_data = preprocessed_data.data

In [43]:
for i in range(len(preprocessed_data["input_ids"])):                    
    inputs = {key: preprocessed_data[key][i].to(device) for key, _ in preprocessed_data.items()}

inputs.keys()


dict_keys(['input_ids', 'token_type_ids', 'attention_mask'])

In [22]:
preprocessed_data.to(device)

outputs = model(**preprocessed_data)

In [23]:
predicted_label_ids = outputs.logits.argmax(dim=1).detach().cpu().numpy()

predicted_labels = [id2label[i] for i in predicted_label_ids]

print(predicted_labels)

['Business', 'Sci/Tech', 'Sports']
